descobrir quais sao os estilos com mais imagens, para depois escolher as classes que o nosso modelo vai dar predict

In [1]:
import csv
from collections import Counter

contagem_estilos = Counter()

with open("bd_original/wikiart_art_pieces.csv", newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)

    for row in reader:
        estilo = row["style"]
        contagem_estilos[estilo] += 1

print("Número de estilos únicos:", len(contagem_estilos))

estilos_ordenados = sorted(
    contagem_estilos.items(),
    key=lambda x: x[1],
    reverse=True
)

print("\nEstilos ordenados por número de imagens:\n")

for estilo, quantidade in estilos_ordenados:
    print(f"{estilo}: {quantidade}")

Número de estilos únicos: 193

Estilos ordenados por número de imagens:

Impressionism: 16083
Realism: 15764
Romanticism: 15010
Expressionism: 11455
Post-Impressionism: 8147
Baroque: 7496
Art Nouveau (Modern): 7382
Unknown: 7039
Surrealism: 6988
Symbolism: 5224
Neoclassicism: 4360
Abstract Expressionism: 3909
Rococo: 3537
Northern Renaissance: 3089
Cubism: 2530
Pop Art: 2517
Academicism: 2474
Minimalism: 2077
Conceptual Art: 1818
Art Informel: 1797
Naïve Art (Primitivism): 1754
Early Renaissance: 1729
Abstract Art: 1729
Magic Realism: 1614
Contemporary Realism: 1599
High Renaissance: 1565
Neo-Expressionism: 1559
Orientalism: 1337
Color Field Painting: 1318
Op Art: 1155
Lyrical Abstraction: 1142
Fauvism: 1106
Contemporary: 992
Neo-Impressionism: 984
Social Realism: 835
Naturalism: 807
Neo-Romanticism: 805
Kitsch: 731
Post-Minimalism: 721
Socialist Realism: 715
Art Deco: 714
Ink and wash painting: 627
Tachisme: 570
Neo-Dada: 568
Hard Edge Painting: 566
Regionalism: 555
Pictorialism: 524


a classe Unknown vai sair

com base no output vamos escolher os top 15 estilos com base no numero de imagens

escolhemos até à classe pop art porque assim temos para cada classe:

    Treino: 1500 imagens por classe.

    Validação: 500 imagens por classe.

    Teste: 500 imagens por classe.

o que isto significa que vamos ter:

    Total de Treino: 22500 imagens.

    Total de Validação: 7500 imagens.

    Total de Teste: 7500 imagens.

escolhidas as classes vamos dividir o dataset corretamente e organizar corretamente a pasta com as imagens

TIVEMOS QUE FAZER UM TESTE DE QUALIDADE ÀS IMAGENS PORQUE HAVIAM IMAGENS CORROMPIDAS. AS IMAGENS CORROMPIDAS NAO VAO ESTAR EM CONSIDERAÇÃO

REPARAMOS QUE HAVIA ESSE PROBLEMA PORQUE QUANDO FOMOS TREINAR PELA PRIMEIRA VEZ DEU UM ERRO: INVALID ARGUMENT ERROR: GRAPH EXECUTION ERROR

In [ ]:
import os
import csv
import random
import shutil
import cv2
from PIL import Image

random.seed(42) 

CLASSES_ESCOLHIDAS = [
    "Impressionism", "Realism", "Romanticism", "Expressionism", "Post-Impressionism",
    "Baroque", "Art Nouveau (Modern)", "Surrealism", "Symbolism", "Neoclassicism",
    "Abstract Expressionism", "Rococo", "Northern Renaissance", "Cubism", "Pop Art"
]

CAMINHO_CSV = os.path.join("bd_original", "wikiart_art_pieces.csv")
PASTA_IMAGENS_ORIGEM = os.path.join("bd_original", "wikiart", "wikiart")
PASTA_DESTINO = "wikiart_dataset"
COLUNA_NOME_FICHEIRO = "file_name"

divisoes = ["train", "validation", "test"]

print("A criar estrutura de pastas...")
for divisao in divisoes:
    for classe in CLASSES_ESCOLHIDAS:
        caminho_pasta = os.path.join(PASTA_DESTINO, divisao, classe)
        os.makedirs(caminho_pasta, exist_ok=True)

imagens_por_classe = {classe: [] for classe in CLASSES_ESCOLHIDAS}

# Função que faz o Teste de Qualidade nas Imagens
def imagem_e_valida(caminho):
    if not os.path.exists(caminho):
        return False
    try:
        # Teste 1: Integridade do cabeçalho
        with Image.open(caminho) as img:
            img.verify()
        # Teste 2: Descodificação dos pixeis
        imagem_cv = cv2.imread(caminho)
        if imagem_cv is None:
            return False
        return True
    except Exception:
        return False

print("A ler o CSV e a verificar cada imagem ...")
imagens_corrompidas_ignoradas = 0

with open(CAMINHO_CSV, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        estilo = row["style"]
        if estilo in CLASSES_ESCOLHIDAS:
            nome_ficheiro = row[COLUNA_NOME_FICHEIRO]
            caminho_origem = os.path.join(PASTA_IMAGENS_ORIGEM, nome_ficheiro)
            
            # Só guardamos o nome se a imagem passar nos 2 testes de qualidade!
            if imagem_e_valida(caminho_origem):
                imagens_por_classe[estilo].append(nome_ficheiro)
            else:
                imagens_corrompidas_ignoradas += 1

print(f"\nVerificação concluída! Foram ignoradas {imagens_corrompidas_ignoradas} imagens corrompidas ou em falta.")
print("A dividir e copiar as imagens válidas...")

for classe in CLASSES_ESCOLHIDAS:
    imagens_validas = imagens_por_classe[classe]
    random.shuffle(imagens_validas)
    
    validacao = imagens_validas[:500]
    teste = imagens_validas[500:1000]
    treino = imagens_validas[1000:2500]
    
    def copiar_lista(lista_imagens, divisao_destino):
        for imagem_nome in lista_imagens:
            origem = os.path.join(PASTA_IMAGENS_ORIGEM, imagem_nome)
            destino = os.path.join(PASTA_DESTINO, divisao_destino, classe, os.path.basename(imagem_nome))
            shutil.copy(origem, destino) 

    copiar_lista(treino, "train")
    copiar_lista(validacao, "validation")
    copiar_lista(teste, "test")
    
    print(f" -> {classe}: {len(treino)} Train | {len(validacao)} Val | {len(teste)} Test")

print("\nDataset organizado com sucesso!")

A criar estrutura de pastas...
A ler o CSV e a verificar cada imagem ...

Verificação concluída! Foram ignoradas 337 imagens corrompidas ou em falta.
A dividir e copiar as imagens válidas...
 -> Impressionism: 1500 Train | 500 Val | 500 Test
 -> Realism: 1500 Train | 500 Val | 500 Test
 -> Romanticism: 1500 Train | 500 Val | 500 Test
 -> Expressionism: 1500 Train | 500 Val | 500 Test
 -> Post-Impressionism: 1500 Train | 500 Val | 500 Test
 -> Baroque: 1500 Train | 500 Val | 500 Test
 -> Art Nouveau (Modern): 1500 Train | 500 Val | 500 Test
 -> Surrealism: 1500 Train | 500 Val | 500 Test
 -> Symbolism: 1500 Train | 500 Val | 500 Test
 -> Neoclassicism: 1500 Train | 500 Val | 500 Test
 -> Abstract Expressionism: 1500 Train | 500 Val | 500 Test
 -> Rococo: 1500 Train | 500 Val | 500 Test
 -> Northern Renaissance: 1500 Train | 500 Val | 500 Test
 -> Cubism: 1500 Train | 500 Val | 500 Test
 -> Pop Art: 1500 Train | 500 Val | 500 Test

Dataset organizado com sucesso!
